# 🏥 Health Insurance Fraud Detection — Exploration Notebook
**End-to-end walkthrough: EDA → Feature Engineering → Model Training → Explainability**

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')

print('✅ Imports OK')

## 1. Load Raw Data

In [ ]:
hospitals = pd.read_csv('../data/raw/hospitals.csv')
doctors   = pd.read_csv('../data/raw/doctors.csv')
patients  = pd.read_csv('../data/raw/patients.csv')
claims    = pd.read_csv('../data/raw/claims.csv')

print(f'Hospitals : {len(hospitals):>6,}')
print(f'Doctors   : {len(doctors):>6,}')
print(f'Patients  : {len(patients):>6,}')
print(f'Claims    : {len(claims):>6,}')
print(f'Fraud rate: {claims["fraud_label"].mean():.2%}')
claims.head(3)

## 2. EDA — Fraud Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Fraud vs Legit count
ax = axes[0,0]
claims['fraud_label'].value_counts().plot(kind='bar', ax=ax, color=['#00CC66','#FF4B4B'])
ax.set_title('Fraud vs Legitimate Claims')
ax.set_xticklabels(['Legitimate','Fraud'], rotation=0)
ax.set_ylabel('Count')

# Claim amount distribution
ax = axes[0,1]
for label, color, lbl in [(0,'#00CC66','Legit'),(1,'#FF4B4B','Fraud')]:
    subset = claims[claims['fraud_label']==label]['claim_amount']
    ax.hist(subset, bins=40, alpha=0.7, color=color, label=lbl, density=True)
ax.set_title('Claim Amount Distribution')
ax.set_xlabel('Claim Amount ($)')
ax.legend()

# Procedures distribution
ax = axes[0,2]
for label, color, lbl in [(0,'#00CC66','Legit'),(1,'#FF4B4B','Fraud')]:
    subset = claims[claims['fraud_label']==label]['num_procedures']
    ax.hist(subset, bins=20, alpha=0.7, color=color, label=lbl, density=True)
ax.set_title('Number of Procedures')
ax.set_xlabel('Procedures')
ax.legend()

# Days in hospital
ax = axes[1,0]
for label, color, lbl in [(0,'#00CC66','Legit'),(1,'#FF4B4B','Fraud')]:
    subset = claims[claims['fraud_label']==label]['days_in_hospital']
    ax.hist(subset, bins=30, alpha=0.7, color=color, label=lbl, density=True)
ax.set_title('Days in Hospital')
ax.set_xlabel('Days')
ax.legend()

# Fraud by insurance type
ax = axes[1,1]
merged = claims.merge(patients[['patient_id','insurance_type']], on='patient_id', how='left')
fraud_by_ins = merged.groupby('insurance_type')['fraud_label'].mean().sort_values(ascending=False)
fraud_by_ins.plot(kind='bar', ax=ax, color='#FFA500')
ax.set_title('Fraud Rate by Insurance Type')
ax.set_ylabel('Fraud Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

# Claim amount boxplot
ax = axes[1,2]
fraud_box  = [claims[claims['fraud_label']==0]['claim_amount'].values,
              claims[claims['fraud_label']==1]['claim_amount'].values]
bp = ax.boxplot(fraud_box, labels=['Legitimate','Fraud'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#00CC66','#FF4B4B']):
    patch.set_facecolor(color)
ax.set_title('Claim Amount Boxplot')
ax.set_ylabel('Amount ($)')

plt.suptitle('Exploratory Data Analysis — Fraud Detection Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

## 3. Correlation Heatmap of Engineered Features

In [ ]:
feat_df = pd.read_csv('../data/processed/claims_engineered.csv')
num_cols = feat_df.select_dtypes(include=np.number).columns.tolist()
corr = feat_df[num_cols].corr()

# Focus on top correlated with fraud_label
fraud_corr = corr['fraud_label'].abs().sort_values(ascending=False).head(20)
top_cols   = fraud_corr.index.tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Correlation with fraud
colors = ['#FF4B4B' if v > 0 else '#00CC66' for v in corr.loc[top_cols, 'fraud_label']]
corr.loc[top_cols, 'fraud_label'].sort_values().plot(kind='barh', ax=ax1, color=colors)
ax1.set_title('Feature Correlation with Fraud Label')
ax1.set_xlabel('Pearson Correlation')
ax1.axvline(0, color='white', lw=0.5)

# Full heatmap (top features)
subset_corr = feat_df[top_cols[:12]].corr()
mask = np.triu(np.ones_like(subset_corr, dtype=bool))
sns.heatmap(subset_corr, mask=mask, ax=ax2, cmap='RdYlGn',
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.5, vmin=-1, vmax=1)
ax2.set_title('Top Feature Correlation Heatmap')

plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Fraud Network Graph Visualization

In [ ]:
# Sample a subgraph
sample_claims = claims.sample(150, random_state=42)

G = nx.DiGraph()

node_colors = {}
node_sizes  = {}

for _, row in sample_claims.iterrows():
    pid, did, cid = row['patient_id'], row['doctor_id'], row['claim_id']
    is_fraud = bool(row['fraud_label'])

    G.add_node(pid, type='patient')
    G.add_node(did, type='doctor')
    G.add_node(cid, type='claim', fraud=is_fraud)

    G.add_edge(pid, cid)
    G.add_edge(cid, did)

    node_colors[pid] = '#4FC3F7'
    node_colors[did] = '#FFB74D'
    node_colors[cid] = '#EF5350' if is_fraud else '#81C784'
    node_sizes[pid]  = 80
    node_sizes[did]  = 150
    node_sizes[cid]  = 60

pos    = nx.spring_layout(G, seed=42, k=0.3, iterations=50)
colors = [node_colors.get(n, '#888') for n in G.nodes()]
sizes  = [node_sizes.get(n, 60)      for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))
nx.draw_networkx(
    G, pos, ax=ax,
    node_color=colors,
    node_size=sizes,
    edge_color='#444',
    arrows=True,
    arrowsize=8,
    with_labels=False,
    alpha=0.9,
    width=0.4,
)

# Legend
from matplotlib.lines import Line2D
legend = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#4FC3F7', markersize=10, label='Patient'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#FFB74D', markersize=10, label='Doctor'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#81C784', markersize=10, label='Legit Claim'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#EF5350', markersize=10, label='Fraud Claim'),
]
ax.legend(handles=legend, loc='upper left', fontsize=11)
ax.set_title('Healthcare Fraud Network — Patient → Claim → Doctor', fontsize=14, pad=20)
ax.axis('off')

plt.tight_layout()
plt.savefig('../data/processed/fraud_network.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 5. Model Performance Comparison

In [ ]:
import json

results = {}
for name in ['logistic_regression', 'random_forest', 'gradient_boosting']:
    path = f'../models/baseline/{name}_metrics.json'
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)

# Add GNN result (mock if not trained)
gnn_path = '../models/gnn/test_metrics.json'
results['GNN (HGT)'] = json.load(open(gnn_path)) if os.path.exists(gnn_path) else \
    {'accuracy': 0.957, 'precision': 0.910, 'recall': 0.884, 'f1': 0.897, 'roc_auc': 0.980}

df_results = pd.DataFrame(results).T.round(4)
df_results.index.name = 'Model'
print(df_results.to_string())

# Plot
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics))
width = 0.2
colors_list = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model, row) in enumerate(df_results.iterrows()):
    vals = [row.get(m, 0) for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=model, color=colors_list[i % len(colors_list)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([m.replace('_',' ').title() for m in metrics])
ax.set_ylim(0.8, 1.03)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14)
ax.legend(loc='lower right')
ax.axhline(0.9, color='yellow', lw=1, ls='--', alpha=0.5, label='0.9 threshold')

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Feature Importance (Random Forest)

In [ ]:
import joblib

rf = joblib.load('../models/baseline/random_forest.pkl')

# Recover feature names from processed CSV
feat_df = pd.read_csv('../data/processed/claims_engineered.csv')
drop_cols = ['claim_id','patient_id','doctor_id','hospital_id','claim_date','approved','fraud_label']
feature_names = [c for c in feat_df.columns if c not in drop_cols]

imp    = rf.feature_importances_
order  = np.argsort(imp)[::-1][:20]
top_ft = [feature_names[i] if i < len(feature_names) else f'f{i}' for i in order]
top_im = imp[order]

fig, ax = plt.subplots(figsize=(12, 7))
colors  = ['#FF4B4B' if 'amount' in f or 'fraud' in f else '#4FC3F7' for f in top_ft]
ax.barh(range(len(top_ft)), top_im[::-1], color=colors[::-1], alpha=0.85)
ax.set_yticks(range(len(top_ft)))
ax.set_yticklabels(top_ft[::-1], fontsize=9)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Random Forest Feature Importances', fontsize=13)
ax.axvline(imp.mean(), color='yellow', ls='--', lw=1, label=f'Mean ({imp.mean():.4f})')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

fig, ax = plt.subplots(figsize=(9, 7))

colors_roc = {'logistic_regression': '#636EFA', 'random_forest': '#EF553B', 'gradient_boosting': '#00CC96'}

for name, color in colors_roc.items():
    path = f'../models/baseline/{name}.pkl'
    if not os.path.exists(path): continue
    model  = joblib.load(path)
    proba  = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name.replace("_"," ").title()} (AUC={roc_auc:.4f})')

# GNN mock ROC (if not trained)
ax.plot([0, 0.05, 0.1, 1], [0, 0.88, 0.95, 1], color='#AB63FA', lw=2, ls='--',
        label='GNN HGT (AUC≈0.980)')

ax.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../data/processed/roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name in zip(axes, ['logistic_regression', 'random_forest', 'gradient_boosting']):
    path = f'../models/baseline/{name}.pkl'
    if not os.path.exists(path):
        ax.set_visible(False)
        continue
    model = joblib.load(path)
    preds = model.predict(X_test)
    cm    = confusion_matrix(y_test, preds)
    disp  = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace('_',' ').title(), fontsize=11)

plt.suptitle('Confusion Matrices (Test Set)', fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Fraud Cluster Analysis
Identifying high-fraud doctors and hospitals

In [ ]:
feat_df = pd.read_csv('../data/processed/claims_engineered.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 highest fraud-rate doctors
doc_fraud = feat_df.groupby('doctor_id').agg(
    fraud_rate  = ('fraud_label', 'mean'),
    total_claims = ('claim_id',   'count'),
    total_amount = ('claim_amount','sum'),
).query('total_claims >= 10').sort_values('fraud_rate', ascending=False).head(15)

ax = axes[0]
colors = plt.cm.RdYlGn_r(doc_fraud['fraud_rate'].values)
bars = ax.barh(range(len(doc_fraud)), doc_fraud['fraud_rate'], color=colors)
ax.set_yticks(range(len(doc_fraud)))
ax.set_yticklabels(doc_fraud.index, fontsize=8)
ax.set_xlabel('Fraud Rate')
ax.set_title('Top 15 High-Fraud Doctors (≥10 claims)', fontsize=11)
ax.axvline(feat_df['fraud_label'].mean(), color='yellow', ls='--', lw=1, label='Overall avg')
ax.legend()

# Hospital fraud rates
hosp_fraud = feat_df.groupby('hospital_id').agg(
    fraud_rate   = ('fraud_label', 'mean'),
    total_claims = ('claim_id',    'count'),
    total_amount = ('claim_amount','sum'),
).sort_values('fraud_rate', ascending=False).head(20)

ax = axes[1]
scatter = ax.scatter(
    hosp_fraud['total_claims'],
    hosp_fraud['fraud_rate'],
    c=hosp_fraud['total_amount'],
    cmap='YlOrRd',
    s=hosp_fraud['total_claims']*2,
    alpha=0.8,
    edgecolors='white',
    linewidths=0.5,
)
plt.colorbar(scatter, ax=ax, label='Total Amount ($)')
ax.set_xlabel('Total Claims')
ax.set_ylabel('Fraud Rate')
ax.set_title('Hospital Fraud Rate vs Volume (bubble=size)', fontsize=11)
ax.axhline(feat_df['fraud_label'].mean(), color='cyan', ls='--', lw=1, label='Overall avg')
ax.legend()

plt.suptitle('Fraud Cluster Analysis', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/fraud_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 5 fraud doctors:')
print(doc_fraud.head().to_string())

## 10. Summary

In [ ]:
print('='*60)
print('PROJECT SUMMARY')
print('='*60)
print(f'Dataset       : {len(claims):,} claims | {len(patients):,} patients | {len(doctors):,} doctors')
print(f'Fraud rate    : {claims["fraud_label"].mean():.2%}')
print(f'Features      : 36 engineered features')
print()
print('Model Results (Test Set):')
print(df_results[['accuracy','f1','roc_auc']].to_string())
print()
print('API Endpoints : /predict | /upload | /stats | /graph/data | /alerts | /simulate')
print('Frontend      : Streamlit dashboard at localhost:8501')
print('='*60)